# Study 929 — Rights Offering — the teardown

Market-model CARs against SPY over three event windows, three placebos (whole-tape, era-matched and clustered), a leave-one-issuer-out jackknife, a discount-band permutation test, an anchor jitter, a timetable sweep, the costed long/short calendar-time book over two holding spans with its beta-adjusted alpha and a borrow sweep, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `9acc9535dfdd`, as-of 2026-06-30).

In [1]:
R = {'start': '2005-01-03', 'end': '2026-06-30', 'n_days': 5406, 'fp': '9acc9535dfdd', 'n_events': 39, 'n_issuers': 20, 'first_deal': '2013-05', 'last_deal': '2023-09', 'ann_mean': 0.27, 'ann_t': 0.34, 'ann_hit': 49, 'ann_lo': -1.2, 'ann_hi': 1.95, 'run_mean': 1.08, 'run_t': 1.2, 'sub_mean': -2.06, 'sub_t': -2.03, 'sub_hit': 41, 'sub_lo': -4.07, 'sub_hi': -0.05, 'post_mean': 0.21, 'post_t': 0.25, 'post_hit': 59, 'post_lo': -1.51, 'post_hi': 1.87, 'deep_n': 18, 'rest_n': 21, 'sub_deep': -2.74, 'sub_rest': -1.47, 'sub_gap': -1.27, 'sub_welch': -0.6, 'sub_perm': 0.53, 'post_deep': -0.09, 'post_rest': 0.48, 'post_gap': -0.57, 'post_welch': -0.31, 'post_perm': 0.751, 'plc_sub_mean': -0.18, 'plc_sub_sd': 2.03, 'plc_sub_z': -0.93, 'plc_sub_p': 0.21, 'plc_ann_z': 0.31, 'plc_ann_p': 0.69, 'plc_post_z': 0.37, 'plc_post_p': 0.68, 'era_plc_sub_sd': 1.28, 'era_plc_sub_z': -1.6, 'era_plc_sub_p': 0.09, 'era_plc_ann_z': 0.46, 'era_plc_ann_p': 0.563, 'era_plc_post_z': 0.25, 'era_plc_post_p': 0.803, 'clu_plc_sub_sd': 2.07, 'clu_plc_sub_z': -0.91, 'clu_plc_sub_p': 0.281, 'clu_plc_ann_z': 0.34, 'clu_plc_ann_p': 0.747, 'clu_plc_post_z': 0.19, 'clu_plc_post_p': 0.768, 'jk_issuer_best': -2.5, 'jk_issuer_best_drop': 'SPE', 'jk_issuer_worst': -1.14, 'jk_issuer_worst_drop': 'CRF', 'era_e_n': 17, 'era_e_mean': -2.65, 'era_e_t': -2.07, 'era_l_n': 22, 'era_l_mean': -1.59, 'era_l_t': -1.05, 'tt28_mean': -1.07, 'tt28_t': -1.43, 'tt40_mean': -2.06, 'tt40_t': -2.03, 'tt55_mean': -1.43, 'tt55_t': -1.29, 'jit_lo_mean': -1.18, 'jit_lo_t': -1.57, 'jit_hi_mean': -2.3, 'jit_hi_t': -1.88, 'long_sharpe': 0.098, 'long_t': 0.44, 'long_cagr': 0.47, 'long_dd': -39.7, 'long_alpha': 0.03, 'long_alpha_t': 0.01, 'long_beta': 0.1, 'long_adv': -0.477, 'long_adv_t': -2.53, 'invested': 25.3, 'avg_names': 1.4, 'short_sharpe': -0.322, 'short_t': -1.43, 'short_cagr': -4.28, 'short_dd': -73.6, 'short_alpha': -2.6, 'short_alpha_t': -1.02, 'cln_long_sharpe': 0.448, 'cln_long_t': 2.03, 'cln_long_cagr': 3.17, 'cln_long_dd': -22.1, 'cln_long_alpha': 2.98, 'cln_long_alpha_t': 1.79, 'cln_long_beta': 0.04, 'cln_long_adv': -0.127, 'cln_long_adv_t': -2.05, 'cln_invested': 13.5, 'cln_avg_names': 1.1, 'cln_short_sharpe': -0.73, 'cln_short_t': -3.24, 'cln_short_dd': -71.4, 'cln_short_alpha': -5.12, 'cln_short_alpha_t': -3.02, 'spy_sharpe': 0.575, 'cost0': 0.168, 'cost10': 0.14, 'cost25': 0.098, 'cost50': 0.028, 'cost100': -0.11, 'borrow0': -0.256, 'borrow100': -0.278, 'borrow300': -0.322, 'borrow800': -0.431, 'cln_borrow0': -0.677, 'cln_borrow800': -0.818, 'syn_ann': -6.62, 'syn_ann_t': -7.88, 'syn_sub': -9.42, 'syn_sub_t': -5.73, 'syn_post': 5.81, 'syn_post_t': 4.96, 'syn_null_post': -0.19, 'syn_null_post_sd': 1.4, 'syn_null_ann': 0.03, 'syn_null_fire': 0, 'dropped_no_tape': 'CUBA, NHF, ENZN, SPLP, TUEM'}

## Design

- **Sample.** 39 rights offerings, 20 issuers, 2013-05 → 2023-09. Hand-compiled anchors, **month precision** (PROXY). Five candidate deals (CUBA, NHF, ENZN, SPLP, TUEM) have no retrievable Yahoo tape — the sample is **survivor-only**, named on the Signal axis.
- **Abnormal return.** `AR_t = r_i,t − (α_i + β_i r_SPY,t)`, with α, β from OLS on the `(−250, −31)` trading-day window before the anchor. Nothing inside any event window enters the estimate.
- **Windows.** announcement `(−1, +5)`, run-up `(−20, −2)`, subscription `(+1, +28)`, post-expiry `(+29, +49)`. The subscription window straddles the modelled ex-rights date and therefore contains **mechanical, un-adjusted dilution**; announcement and post-expiry are clean.
- **One execution lag, and the hole in it.** Every tradable leg enters at the close of the session *after* the anchor; the announcement-day return is an event-study quantity only. But the anchors are month-precision, so for roughly half the deals the anchor precedes the true press release and the book is long before the news is public — a **genuine look-ahead channel**, named here. It can only manufacture a reaction a real-time trader would have missed, so it biases the study *towards* an effect; the ±10-day jitter sweeps it.
- **Total return.** `auto_adjust=True` throughout — vital for CEFs, which distribute most of their return.
- **Assumptions swept.** The rights timetable (ex-rights +10d, expiry +40d) and the `deep`/`moderate`/`shallow` band labels.

## The event-window table

In [2]:
print(f"run-up       (-20,-2)  mean {R['run_mean']:+6.2f}%  t {R['run_t']:+6.2f}")
print(f"announcement ( -1,+5)  mean {R['ann_mean']:+6.2f}%  t {R['ann_t']:+6.2f}  "
      f"hit {R['ann_hit']}%  boot95 [{R['ann_lo']:+.2f}%, {R['ann_hi']:+.2f}%]")
print(f"subscription ( +1,+28) mean {R['sub_mean']:+6.2f}%  t {R['sub_t']:+6.2f}  "
      f"hit {R['sub_hit']}%  boot95 [{R['sub_lo']:+.2f}%, {R['sub_hi']:+.2f}%]  <- dilution mixed in")
print(f"post-expiry  (+29,+49) mean {R['post_mean']:+6.2f}%  t {R['post_t']:+6.2f}  "
      f"hit {R['post_hit']}%  boot95 [{R['post_lo']:+.2f}%, {R['post_hi']:+.2f}%]")

run-up       (-20,-2)  mean  +1.08%  t  +1.20
announcement ( -1,+5)  mean  +0.27%  t  +0.34  hit 49%  boot95 [-1.20%, +1.95%]
subscription ( +1,+28) mean  -2.06%  t  -2.03  hit 41%  boot95 [-4.07%, -0.05%]  <- dilution mixed in
post-expiry  (+29,+49) mean  +0.21%  t  +0.25  hit 59%  boot95 [-1.51%, +1.87%]


## Three placebos — and why the flattering one is not the one to quote

The naive cross-sectional *t* treats 39 deals as 39 independent draws. Brown & Warner (1985) and Kolari & Pynnönen (2010) say it is not, so we resample. But a placebo is itself a modelling choice, and the obvious version is rigged in our favour:

- **A — whole-tape anchors.** 300 draws, same tickers, dates uniform over 2005-2026. That range hands the null the GFC and the COVID crash, regimes in which **no deal on this list was announced**. Its dispersion is inflated and it makes our observed drift look tame.
- **B — era-matched anchors.** Same, but drawn only from 2012-2024, the years the deals actually happened. **This is the fair yardstick and the one the verdict quotes.**
- **C — clustered.** Slide the *whole list* by one random offset, so every gap between deals survives and the cross-event dependence (both Cornerstone funds every September, the Gabelli funds every April) is preserved. This is the one that actually answers the Brown & Warner objection; the wide range means it inherits A's regime problem too.

In [3]:
print(f"observed subscription CAR: {R['sub_mean']:+.2f}%  "
      f"(naive parametric SE {abs(R['sub_mean']/R['sub_t']):.2f}%)\n")
print(f"{'placebo':28s}{'sd':>7s}{'z':>8s}{'p':>8s}")
print(f"{'A whole tape 2005-2026':28s}{R['plc_sub_sd']:6.2f}%"
      f"{R['plc_sub_z']:+8.2f}{R['plc_sub_p']:8.3f}   <- flattering")
print(f"{'B era-matched 2012-2024':28s}{R['era_plc_sub_sd']:6.2f}%"
      f"{R['era_plc_sub_z']:+8.2f}{R['era_plc_sub_p']:8.3f}   <- quote this one")
print(f"{'C clustered common shift':28s}{R['clu_plc_sub_sd']:6.2f}%"
      f"{R['clu_plc_sub_z']:+8.2f}{R['clu_plc_sub_p']:8.3f}")
print()
print(f"announcement, era-matched : z {R['era_plc_ann_z']:+.2f}, "
      f"p {R['era_plc_ann_p']:.3f}")
print(f"post-expiry,  era-matched : z {R['era_plc_post_z']:+.2f}, "
      f"p {R['era_plc_post_p']:.3f}")
print()
print(f"-> the fair placebo still does not clear the drift "
      f"(p = {R['era_plc_sub_p']:.2f}), but it is 0.09, not 0.21:")
print('   the placebo is a supporting witness here, not the case. The case is')
print('   that the drift needs one issuer, one era and one guessed date to exist.')

observed subscription CAR: -2.06%  (naive parametric SE 1.01%)

placebo                          sd       z       p
A whole tape 2005-2026        2.03%   -0.93   0.210   <- flattering
B era-matched 2012-2024       1.28%   -1.60   0.090   <- quote this one
C clustered common shift      2.07%   -0.91   0.281

announcement, era-matched : z +0.46, p 0.563
post-expiry,  era-matched : z +0.25, p 0.803

-> the fair placebo still does not clear the drift (p = 0.09), but it is 0.09, not 0.21:
   the placebo is a supporting witness here, not the case. The case is
   that the drift needs one issuer, one era and one guessed date to exist.


## Robustness on the subscription window

Leave-one-issuer-out (the list has 39 deals from 20 issuers), the era cut, the assumed timetable, and the ±10 calendar-day anchor jitter.

In [4]:
print(f"leave-one-issuer-out: t from {R['jk_issuer_best']:+.2f} (drop {R['jk_issuer_best_drop']}) "
      f"to {R['jk_issuer_worst']:+.2f} (drop {R['jk_issuer_worst_drop']})")
print(f"era 2013-2017 (n={R['era_e_n']}): {R['era_e_mean']:+.2f}% (t={R['era_e_t']:+.2f})   "
      f"era 2018-2023 (n={R['era_l_n']}): {R['era_l_mean']:+.2f}% (t={R['era_l_t']:+.2f})")
print(f"timetable expiry 28d {R['tt28_mean']:+.2f}% (t={R['tt28_t']:+.2f})  "
      f"40d {R['tt40_mean']:+.2f}% (t={R['tt40_t']:+.2f})  "
      f"55d {R['tt55_mean']:+.2f}% (t={R['tt55_t']:+.2f})")
print(f"anchor jitter +/-10d: mean spans {R['jit_hi_mean']:+.2f}% to {R['jit_lo_mean']:+.2f}%, "
      f"t spans {R['jit_hi_t']:+.2f} to {R['jit_lo_t']:+.2f} -> never robustly past |2|")

leave-one-issuer-out: t from -2.50 (drop SPE) to -1.14 (drop CRF)
era 2013-2017 (n=17): -2.65% (t=-2.07)   era 2018-2023 (n=22): -1.59% (t=-1.05)
timetable expiry 28d -1.07% (t=-1.43)  40d -2.06% (t=-2.03)  55d -1.43% (t=-1.29)
anchor jitter +/-10d: mean spans -2.30% to -1.18%, t spans -1.88 to -1.57 -> never robustly past |2|


## Is the discount compensated? Band split + label permutation

In [5]:
print(f"subscription: deep (n={R['deep_n']}) {R['sub_deep']:+.2f}%  vs rest (n={R['rest_n']}) "
      f"{R['sub_rest']:+.2f}%  gap {R['sub_gap']:+.2f}%  Welch t {R['sub_welch']:+.2f}  "
      f"perm p {R['sub_perm']:.3f}")
print(f"post-expiry : deep {R['post_deep']:+.2f}%  vs rest {R['post_rest']:+.2f}%  "
      f"gap {R['post_gap']:+.2f}%  Welch t {R['post_welch']:+.2f}  perm p {R['post_perm']:.3f}")
print('\n-> the band is an ASSUMPTION and it buys us nothing: a random relabelling')
print('   reproduces both gaps more than half the time.')

subscription: deep (n=18) -2.74%  vs rest (n=21) -1.47%  gap -1.27%  Welch t -0.60  perm p 0.530
post-expiry : deep -0.09%  vs rest +0.48%  gap -0.57%  Welch t -0.31  perm p 0.751

-> the band is an ASSUMPTION and it buys us nothing: a random relabelling
   reproduces both gaps more than half the time.


## Tradability — calendar-time book, excess-of-cash on both legs, one lag

Equal-weight every name inside the holding span, 25 bps one-way × NAV on entry and exit, flat days in BIL. Two spans, because the obvious one is contaminated:

- **full `(+1, +49)`** — spans the modelled ex-rights date, so the long leg eats a price fall a real subscriber was compensated for and the short leg pockets it. It **understates** the long book.
- **clean `(+29, +49)`** — after the modelled expiry, no dilution artefact. This is the discount's best case, and we report it as such.

Both legs are excess-of-cash; the short leg is **credited its cash collateral** and pays borrow on top (charging borrow while withholding the rebate would be a one-sided cost). `alpha` is the beta-adjusted intercept vs SPY with a HAC *t* — the statistic to read, because the book is long-only equity 13.5–25.3% of the time and its **own** *t* mostly prices the beta it rents, while the vs-SPY Sharpe race is exposure-mismatched the other way.

In [6]:
print(f"{'leg / span':26s}{'exSharpe':>10s}{'own t':>8s}{'alpha/yr':>10s}"
      f"{'t(a)':>7s}{'beta':>7s}{'DD':>8s}")
for tag, s, t, a, at, b, dd in [
    ('long  full  (+1,+49)', R['long_sharpe'], R['long_t'], R['long_alpha'],
     R['long_alpha_t'], R['long_beta'], R['long_dd']),
    ('long  clean (+29,+49)', R['cln_long_sharpe'], R['cln_long_t'],
     R['cln_long_alpha'], R['cln_long_alpha_t'], R['cln_long_beta'], R['cln_long_dd']),
    ('short full  (+1,+49)', R['short_sharpe'], R['short_t'], R['short_alpha'],
     R['short_alpha_t'], -R['long_beta'], R['short_dd']),
    ('short clean (+29,+49)', R['cln_short_sharpe'], R['cln_short_t'],
     R['cln_short_alpha'], R['cln_short_alpha_t'], -R['cln_long_beta'],
     R['cln_short_dd']),
]:
    print(f'{tag:26s}{s:+10.3f}{t:+8.2f}{a:+9.2f}%{at:+7.2f}{b:+7.2f}{dd:+8.1f}')
print(f"{'SPY (excess of cash)':26s}{R['spy_sharpe']:+10.3f}")
print(f"\ncost sweep (long full, one-way bps): 0 {R['cost0']:+.3f} | 10 {R['cost10']:+.3f} | "
      f"25 {R['cost25']:+.3f} | 50 {R['cost50']:+.3f} | 100 {R['cost100']:+.3f}")
print(f"borrow sweep (short full, bps/yr) : 0 {R['borrow0']:+.3f} | 100 {R['borrow100']:+.3f} | "
      f"300 {R['borrow300']:+.3f} | 800 {R['borrow800']:+.3f}")
print(f"borrow sweep (short clean, bps/yr): 0 {R['cln_borrow0']:+.3f} "
      f"... 800 {R['cln_borrow800']:+.3f}")
print('\n-> the clean long book is the strongest positive in this study and it is')
print('   still not an edge: Sharpe +0.448 is rented beta, alpha +2.98%/yr t=+1.79,')
print('   on a window chosen AFTER seeing the full span was contaminated, and it')
print('   loses the race to simply owning SPY.')

leg / span                  exSharpe   own t  alpha/yr   t(a)   beta      DD
long  full  (+1,+49)          +0.098   +0.44    +0.03%  +0.01  +0.10   -39.7
long  clean (+29,+49)         +0.448   +2.03    +2.98%  +1.79  +0.04   -22.1
short full  (+1,+49)          -0.322   -1.43    -2.60%  -1.02  -0.10   -73.6
short clean (+29,+49)         -0.730   -3.24    -5.12%  -3.02  -0.04   -71.4
SPY (excess of cash)          +0.575

cost sweep (long full, one-way bps): 0 +0.168 | 10 +0.140 | 25 +0.098 | 50 +0.028 | 100 -0.110
borrow sweep (short full, bps/yr) : 0 -0.256 | 100 -0.278 | 300 -0.322 | 800 -0.431
borrow sweep (short clean, bps/yr): 0 -0.677 ... 800 -0.818

-> the clean long book is the strongest positive in this study and it is
   still not an edge: Sharpe +0.448 is rented beta, alpha +2.98%/yr t=+1.79,
   on a window chosen AFTER seeing the full span was contaminated, and it
   loses the race to simply owning SPY.


## Live synthetic control — the harness is unbiased

**Synthetic data, not the real tape.** Planted world: an announcement drop, a subscription slide and a post-expiry bounce (1.5× for the deep band) that the event study must recover. Null world: identical anchors, effect switched off — the study must stay quiet, and must stay quiet *across seeds*, since any single seed can throw a 2-sigma window.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from rights_offering import data, strategy as st
import numpy as np
px, ev, truth = data.synthetic_panel(signal_strength=1.0, seed=929)
d = st.synthetic_detect(px, ev)
print('planted (n=%d): announce %+.2f%% (t=%+.2f)  subscription %+.2f%% (t=%+.2f)  '
      'post-expiry %+.2f%% (t=%+.2f)'
      % (d['n'], d['announce_mean_pct'], d['announce_t'],
         d['subscription_mean_pct'], d['subscription_t'],
         d['post_expiry_mean_pct'], d['post_expiry_t']))
post, ann, ts = [], [], []
for s in range(12):
    p0, e0, _ = data.synthetic_panel(signal_strength=0.0, seed=929 + s)
    d0 = st.synthetic_detect(p0, e0)
    post.append(d0['post_expiry_mean_pct']); ann.append(d0['announce_mean_pct'])
    ts.append(d0['post_expiry_t'])
post, ann, ts = np.array(post), np.array(ann), np.array(ts)
print('null x12: post-expiry mean %+.2f%% (sd %.2f), announce mean %+.2f%%, '
      '|t|>=2 in %d/12' % (post.mean(), post.std(ddof=1), ann.mean(), (abs(ts) >= 2).sum()))

planted (n=37): announce -6.62% (t=-7.88)  subscription -9.42% (t=-5.73)  post-expiry +5.81% (t=+4.96)


null x12: post-expiry mean -0.19% (sd 1.40), announce mean +0.03%, |t|>=2 in 0/12


## Discount-band recovery on the planted world

**Synthetic.** Deep-band names carry 1.5× the planted effect, so the same split that finds nothing on the real tape must find something here.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from rights_offering import data, strategy as st
px, ev, _ = data.synthetic_panel(signal_strength=1.0, seed=929)
panel = st.event_panel(px, ev)
sp = st.discount_split(panel, 'post_expiry')
pm = st.permutation_discount_test(panel, 'post_expiry', n_perm=2000, seed=929)
print('SYNTHETIC deep %+.2f%% vs rest %+.2f%%  gap %+.2f%% (Welch t %+.2f), perm p %.3f'
      % (sp['mean_deep_pct'], sp['mean_rest_pct'], sp['diff_pct'],
         sp['welch_t'], pm['p_two_sided']))
print('-> the split works when there is something to find; on the real tape p = %.2f'
      % 0.751)

SYNTHETIC deep +10.17% vs rest +3.45%  gap +6.72% (Welch t +2.67), perm p 0.005
-> the split works when there is something to find; on the real tape p = 0.75


## Verdict

- **Signal — None.** Announcement +0.27% (*t* = +0.34, bootstrap [-1.20%, +1.95%]); post-expiry +0.21% (*t* = +0.25, [-1.51%, +1.87%]); discount-band gap -0.57% with permutation *p* = 0.75. The lone |*t*| ≥ 2 — the subscription window's -2.06% (*t* = -2.03) — sits in the window carrying **mechanical, un-adjusted ex-rights dilution** and fails every robustness cut: the fair era-matched placebo *z* = -1.60 (*p* = 0.09; the flattering whole-tape version says 0.21 and we do not lean on it), leave-one-issuer-out *t* → -1.14, recent era *t* = -1.05, timetable *t* ∈ [-1.43, -2.03], anchor jitter never clearing |2| with room. The synthetic control recovers a planted effect (announce -6.62%, *t* = -7.88) and is centred on the null (-0.19%, sd 1.40, 0/12 firings), so the silence is real.
- **Tradability — Mirage.** On the full span the long book earns excess Sharpe +0.098 with a beta-adjusted alpha of +0.03%/yr (HAC *t* +0.01) — and that span is penalised by the dilution artefact. Give the discount its best case on the clean `(+29,+49)` span and the Sharpe is +0.448, but the alpha is +2.98%/yr (HAC *t* +1.79) on a book live 13.5% of days averaging 1.1 names through a -22.1% drawdown: rented beta, not edge, and still behind SPY's +0.575. The short leg is negative on both spans at every borrow rate (-0.256 to -0.431 full, -0.677 to -0.818 clean).
- **Limits, stated plainly.** A month-precision, hand-compiled, survivor-only list of 39 deals from 20 issuers can rule out a *large* rights-offering effect; it cannot rule out a small one. Month precision also means the book is sometimes long *before* the press release — a look-ahead we name rather than hide. All three biases we know of (that look-ahead, survivorship, and the dilution contaminating the only negative window) push **towards** finding an effect, and we still find none.